In [10]:
from pymilvus import (
    connections, FieldSchema, CollectionSchema, DataType, Collection, utility
)
import numpy as np
import pandas as pd

from nltk.corpus import stopwords
import spacy
import nltk
import json
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

connections.connect("default", host="localhost", port="19530")
# nltk.download('stopwords')
# spacy.cli.download("en_core_web_sm")

In [ ]:
# fields = [
#     FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
#     FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=384),
#     FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=65535),
#     FieldSchema(name="year", dtype=DataType.INT64),
#     FieldSchema(name="month", dtype=DataType.INT64),
#     FieldSchema(name="day", dtype=DataType.INT64),
#     FieldSchema(
#         name="words",
#         dtype=DataType.ARRAY,
#         element_type=DataType.VARCHAR,
#         max_length=400,
#         max_capacity=4096
#     ),
#     FieldSchema(
#         name="entity_names",
#         dtype=DataType.ARRAY,
#         element_type=DataType.VARCHAR,
#         max_length=300,
#         max_capacity=600
#     ),
#     FieldSchema(
#         name="entity_ids",
#         dtype=DataType.ARRAY,
#         element_type=DataType.VARCHAR,
#         max_length=300,
#         max_capacity=600
#     ),
# ]


# schema = CollectionSchema(fields=fields, description="Collection of Russian Speeches")

# collection_name = "russian_speeches"
# if collection_name in utility.list_collections():
#     utility.drop_collection(collection_name)

# collection = Collection(name=collection_name, schema=schema)
# collection.create_index(
#     field_name="embedding",
#     index_params={
#         "index_type": "IVF_FLAT",
#         "metric_type": "COSINE",
#         "params": {"nlist": 1024}
#     }
# )
# print(f"Collection `{collection_name}` created successfully.")

Collection `russian_speeches` created successfully.


In [12]:
collection = Collection("russian_speeches")
collection.load()

In [13]:
with open("data/putin_complete.json", "r") as f:
    speeches = json.load(f)

stop_words = set(stopwords.words('english'))
punctuation = [".", ",", "?", "!", ":", "`", "'", "(", ")", "[", "]", "/", '’', "-", "’s", "\"", ";", "i", " ", "–", "%", "*", "...", "…"]
lemmatize = spacy.load("en_core_web_sm")
model = SentenceTransformer("all-MiniLM-L6-v2")

data = [[],[],[],[],[],[],[],[]]

for speech in tqdm((speeches), "Populating collection..."):
    
    text = speech["transcript_filtered"]
    if len(text) > 20000:
        continue
    
    doc = lemmatize(text)

    splitted_date = speech["date"].split("-")
    year = int(splitted_date[0])
    month = int(splitted_date[1])
    day = int(splitted_date[2].split("T")[0])

    words = [
        w.lemma_.lower() for w in doc if not (w.lemma_ in stop_words or w.lemma_.lower() in punctuation or " " in w.lemma_)
    ]

    entity_names = []
    entity_ids = []

    for ent in doc.ents:
        entity_names.append(ent.text)
        entity_ids.append(f"{ent.start},{ent.end}")
    # data = [[],[],[],[],[],[],[],[]]
    data[0].append(model.encode(text))
    data[1].append(text)
    data[2].append(year)
    data[3].append(month)
    data[4].append(day)
    data[5].append(words)
    data[6].append(entity_names)
    data[7].append(entity_ids)

    # try:
    #     collection.insert(data)
    # except:
    #     print(f"speech from {year}-{month}-{day} skipped.")
    #     continue
collection.insert(data)

Populating collection...: 100%|██████████| 9838/9838 [19:36<00:00,  8.36it/s] 
2025-09-08 18:51:59,867 [ERROR][handler]: grpc RpcError: [batch_insert], <_MultiThreadedRendezvous: StatusCode.RESOURCE_EXHAUSTED, grpc: received message larger than max (87931304 vs. 67108864)>, <Time:{'RPC start': '2025-09-08 18:51:57.580659', 'gRPC error': '2025-09-08 18:51:59.866202'}> (decorators.py:151)


_MultiThreadedRendezvous: <_MultiThreadedRendezvous of RPC that terminated with:
	status = StatusCode.RESOURCE_EXHAUSTED
	details = "grpc: received message larger than max (87931304 vs. 67108864)"
	debug_error_string = "UNKNOWN:Error received from peer ipv6:%5B::1%5D:19530 {grpc_status:8, grpc_message:"grpc: received message larger than max (87931304 vs. 67108864)"}"
>

In [14]:
results = collection.query(
    expr="year == 2000",
    output_fields=["year", "month", "day"]
)
results

data: [], extra_info: {}